# STAC Client Implementation Outline

## Client Search Scenario

Connected Data Asset STAC clients are, at their core, just conventional STAC clients respecting a number of Best Practices to make the STAC API protocol useful in the WGISS environment. The collection identifier of the collection of interest is a mandatory element in a CEOS STAC request to find granules (a.k.a. STAC items). Clients could retrieve collection IDs from the IDN or FedEO Collection search responses.

This chapter will give brief steps about how to retrieve a collection ID and how to interact with the granule search server for inventory search. The corresponding details are elaborated in the Use Case chapters.

We assume that Data providers have registered the metadata of their archived collections into the IDN or have made their collections accessible to FedEO. The client can query the IDN or FedEO to retrieve the collection ID for a desired collection of interest and, based on that collection ID and other spatial-temporal query conditions, build a valid CEOS STAC query. The following steps describe the client search scenario starting with a STAC Collection search.

In [158]:
%pip install geopandas
# %pip install --force-reinstall -v "pystac_client==0.6.1"
%pip install pystac_client==0.8.5
# %pip install intake

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pystac_client==0.8.5 in c:\users\yvesc\appdata\local\programs\python\python312\lib\site-packages (0.8.5)




[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [159]:
%pip install folium matplotlib mapclassify
%pip install jsonpath_ng

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [160]:
import folium
import folium.plugins
import geopandas as gpd
import shapely.geometry
import pandas as pd  
import numpy as np
import json
import urllib.parse
import requests

from xml.dom import minidom
from IPython.display import HTML, display
from IPython.display import Markdown as md
from pystac_client import Client
from pystac import Collection
from typing import Any, Dict
from urllib.parse import urlparse, parse_qsl
from matplotlib import pyplot as plt, cm, colors
from PIL import Image
from io import BytesIO
from branca.element import Figure
from concurrent.futures import ThreadPoolExecutor

import warnings
#warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', category=UserWarning)


def convert_bounds(bbox, invert_y=False):
    """
    Helper method for changing bounding box representation to leaflet notation

    ``(lon1, lat1, lon2, lat2) -> ((lat1, lon1), (lat2, lon2))``
    """
    x1, y1, x2, y2 = bbox
    if invert_y:
        y1, y2 = y2, y1
    return ((y1, x1), (y2, x2))

def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False
    

def curl_command( url: str, method: str = "GET" ) -> str:
    """
    Convert request URL to equivalent curl GET or POST command-line
    for STAC search (bash shell).
    """
    c = "curl -X " + method
    res = urlparse(url)
    if "GET" in method:
        c = c + " -G " + res.scheme + "://" + res.netloc + res.path
    else:
        c = c + " " + res.scheme + "://" + res.netloc + res.path    \
              + " \\\n\t--header 'Content-Type: application/json'"  \
              + " \\\n\t--data-raw '{"
    
    lst = parse_qsl(res.query)
    
    first = True 
    for i in lst:
        # print(i[0])
        # add \ to end of previous line
        if "GET" in method:
            # correction 16/3: data-urlencode used.
            v = i[1].replace('"','\\"')
            c = c + ' \\\n\t--data-urlencode "'+i[0]+'='+v+'"'
        else:
            if not(first):
                c = c + ','
                
            if i[0] in ["ids","collections"]:
                # "collections" and "ids") parameter has to be included as an array.
                lst = i[1].split(',')       
                c = c + '\n\t\t"'+i[0]+'": '+str(lst).replace("'","\"")
            
            elif is_number(i[1]) or i[1][0]=='{' or i[1][0]=='[':
                # do not surround with quotes if numerical value  or a json object or an array
                c = c + '\n\t\t"'+i[0]+'": '+i[1]
            else:
                c = c + '\n\t\t"'+i[0]+'": "'+i[1]+'"'
        first = False
        
    if "POST" in method:
        c = c + "\n\t}'"
    return c


def display_previews(results):
    """
    Helper method for displaying a grid of quicklooks (if available)
    """
    # create figure
    fig = plt.figure(figsize=(20, 20))
  
    # setting values to rows and column variables for the image grid
    rows = 8
    columns = 2
    pos = 1

    for item in results.items():
        # print(item.id)
        assets = item.assets
        try:
            # print("found thumbnail", assets['thumbnail'].href)
            url = assets['thumbnail'].href
            response = requests.get(url)
            Image1 = Image.open(BytesIO(response.content))
            # display at position 'pos'
            fig.add_subplot(rows, columns, pos)
            pos = pos+1
            # show the image
            plt.imshow(Image1)
            plt.axis('off')
            plt.title(item.id)
        except:
            pass
    return

def display_gdf_plot(results):
    """
    Helper method for displaying results as dataframe plot
    """
    # https://github.com/opendatacube/odc-stac (License Apache 2.0)
    # https://odc-stac.readthedocs.io/en/latest/notebooks/stac-load-e84-aws.html#Plot-STAC-Items-on-a-Map

    # Convert STAC items into a GeoJSON FeatureCollection
    stac_json = results.item_collection_as_dict()

    gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")

    fig = gdf.plot(
        "datetime",
        edgecolor="black",
        categorical=True,
        aspect="equal",
        alpha=0.5,
        figsize=(6, 12),
        legend=True,
        legend_kwds={"loc": "upper left", "frameon": False, "ncol": 1},
    )
    _ = fig.set_title("STAC Query Results")

    # gdf
    return

def display_date_distribution(results):
    """
    Helper method for displaying number of results per year/month as bar chart
    """
    items = list(results.items())
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json)

    gdf['date'] = pd.to_datetime(gdf['start_datetime'])
    # create a representation of the month with strfmt
    gdf['year_month'] = gdf['date'].map(lambda dt: dt.strftime('%Y-%m'))
    grouped_df = gdf.groupby('year_month')['year_month'].size().to_frame("count").reset_index()
    grouped_df.plot(kind='bar', x='year_month', y='count')
    return

def display_value_distribution(results, column):
    """
    Helper method for displaying number values in column as bar chart
    """
    items = list(results.items())
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json)

    # gdf['date'] = pd.to_datetime(gdf['start_datetime'])
    # create a representation of the month with strfmt
    # gdf['year_month'] = gdf['date'].map(lambda dt: dt.strftime('%Y-%m'))

    try:
        grouped_df = gdf.groupby(column)[column].size().to_frame("count").reset_index()
        grouped_df.plot(kind='bar', x=column, y='count')
    except:
        print(column + " values are not available.")

    return


def display_map(results):
    """
    Helper method for displaying results on a map
    """
    # https://github.com/python-visualization/folium/issues/1501
    stac_json = results.item_collection_as_dict()
    gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")
    
    fig = Figure(width="800px", height="500px")
    map1 = folium.Map()
    fig.add_child(map1)

    # folium.GeoJson(
    #    shapely.geometry.box(*bbox),  # ??
    #    style_function=lambda x: dict(fill=False, weight=1, opacity=0.7, color="olive"),
    #    name="Query",
    # ).add_to(map1)

    gdf.explore(
        "start_datetime",
        categorical=True,
        tooltip=[
            "title", "datetime", "start_datetime", "platform", "instruments"    
        ],
        popup=True,
        style_kwds=dict(fillOpacity=0.1, width=2),
        name="STAC",
        m=map1,
    )

    # map1.fit_bounds(bounds=convert_bounds(gdf.unary_union.bounds))
    map1.fit_bounds(bounds=convert_bounds(gdf.union_all().bounds))
    display(fig)
    return


In [161]:
URL_LANDING_PAGE =  'https://fedeo.ceos.org/' 

In [162]:
COLLECTION_ID1 = 'PROBA.CHRIS.1A'
COLLECTION_ID2 = 'SPOT-6.and.7.ESA.archive'  
COLLECTION_ID2_CLOUDS = 'LANDSAT.ETM.GTC'
COLLECTION_ID3_CLOUDS = 'IKONOS.ESA.archive'
COLLECTION_ID4 = 'Deimos-1.and.2.ESA.archive' 
COLLECTION_ID5_DMSMM = 'NOAA_AVHRR_L1B_LAC'  # collection with Data Mgt and Stewardship Maturity Matrix


**Step 1**  
>  Obtain the STAC API Collection search endpoint to formulate a valid Collection search request.

The API implements the STAC API Collection Search Extension [[RD25]](#RD25).  The URL of the collection search endpoint is advertized in the Landing Page as a link (rel="data").
The code below extracts this link from the FedEO landing page.

In [163]:
from jsonpath_ng.ext import parse

response = requests.get(URL_LANDING_PAGE)
data = json.loads(response.text)
expression = parse("$.links[?(@.rel == 'data')].href")
r = expression.find(data)
r[0].value

'https://fedeo.ceos.org/collections'

The link with rel="http://www.opengis.net/def/rel/ogc/1.0/queryables" provides access to the list of filter criteria available for collection search.  It returns a Queryables object in JSON Schema format.
The list of filter criteria IDN and FedEO support for collection searches differs.

In [164]:
# retrieve /collections response
response = requests.get(r[0].value)
data = json.loads(response.text)

In [165]:
from jsonpath_ng.ext import parse

expression = parse("$.links[?(@.rel == 'http://www.opengis.net/def/rel/ogc/1.0/queryables')].href")
r = expression.find(data)
r[0].value

'https://fedeo.ceos.org/collections/queryables'

In [166]:
# Get queryables response and list parameters alphabetically.
response = requests.get(r[0].value)
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

,key,type,format
14,classifiedAs,string,uri
13,doi,string,-
3,instrument,string,-
8,modificationDate,string,date-time
11,offering,string,-
7,organisationName,string,-
5,otherConstraint,string,-
2,parentIdentifier,string,-
6,platform,string,-
9,processingLevel,string,-


**Step 2**  
>  Search collections of interest through STAC with proper request parameters (e.g. spatial footprint, temporal extent and keyword).

The collection search supports free text searches using the `q` query parameter as shown below.

In [167]:
from pystac_client import Client
api = Client.open(URL_LANDING_PAGE)

results = api.collection_search(
    q = 'Proba-1'
)

In [168]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://fedeo.ceos.org/collections \
	--data-urlencode "q=Proba-1"
```


In [169]:
data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=['collections'])
df[['id', 'keywords']]

,id,keywords
0,PROBA.CHRIS.1A,"[DIF10, EARTH SCIENCE>BIOSPHERE>ECOSYSTEMS>TER..."
1,PROBA.HRC.1A,"[DIF10, EARTH SCIENCE>BIOSPHERE>ECOSYSTEMS>TER..."


**Step 3**  
>  From the collection search response, obtain the granule search endpoint for this collection, or obtain the collection ID to be used in a cross-collection search request.

The cross collection granule search endpoint is advertized in the Landing Page as a link with rel="search".

In [170]:
from jsonpath_ng.ext import parse

# find the link with rel="search" and method="GET"

response = requests.get(URL_LANDING_PAGE)
data = json.loads(response.text)
expression = parse("$.links[?(@.method == 'GET' & @.rel == 'search' )].href")
r = expression.find(data)
r[0].value


'https://fedeo.ceos.org/search'

The granule search endpoint for the individual collection is advertized in the collection metadata as a link with rel="items".  The collection metadata can be found inside a collection search response or be accessed directly if its collection ID is known.  The server may support both the `GET` and `POST` methods.

In [171]:
md(f"For example, the collection metadata for `{COLLECTION_ID1}`, is available at  \
at {URL_LANDING_PAGE + 'collections/' + COLLECTION_ID1}.  This corresponds to one of the many representations available using content-negotiation.")

For example, the collection metadata for `PROBA.CHRIS.1A`, is available at  at https://fedeo.ceos.org/collections/PROBA.CHRIS.1A.  This corresponds to one of the many representations available using content-negotiation.

In [172]:
URL = URL_LANDING_PAGE + 'collections/' + COLLECTION_ID1

In [173]:
curl_str = curl_command(URL)
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://fedeo.ceos.org/collections/PROBA.CHRIS.1A
```


In [174]:
# retrieve collection by identifier and extract granule search endpoint
response = requests.get(URL)
data_collections = json.loads(response.text)

expression = parse("$.links[?(@.rel == 'items' )].href")
r = expression.find(data_collections)
r[0].value

'https://fedeo.ceos.org/collections/PROBA.CHRIS.1A/items?httpAccept=application/geo%2Bjson;profile=https://stacspec.org'

**Step 4**  
>  Based on the common query paramters or on retrieved Queryables, formulate a STAC API search request for granules belonging to that collection, directed to the cross-collection search endpoint.

In [175]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',         
    max_items = 2,
    collections=[COLLECTION_ID1],
    bbox = [14.90, 37.700, 14.99, 37.780], # Mount Etna
    datetime=['2015-01-01T00:00:00Z', '2022-01-02T00:00:00Z']
)

In [176]:
curl_str = curl_command(results.url_with_parameters(), "GET")
md("```shell\n" + curl_str + "\n```\n")

```shell
curl -X GET -G https://fedeo.ceos.org/search \
	--data-urlencode "bbox=14.9,37.7,14.99,37.78" \
	--data-urlencode "datetime=2015-01-01T00:00:00Z/2022-01-02T00:00:00Z" \
	--data-urlencode "collections=PROBA.CHRIS.1A"
```


In [177]:
print(f"{results.matched()} items found.")

9 items found.


In [ ]:
# Show search response (GeoJSON)
data = results.item_collection_as_dict()
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")


## Obtaining the Queryables Document

A Queryables document provides necessary information for clients to programmatically formulate valid search requests using the `filter` parameter. Specifically, clients are expected to acquire both the name and type of the parameters based on the JSON Schema returned by the `/queryables` endpoint. 

The list of supported filter parameters (a.k.a. queryables) for this collection can be retrieved from the collection metadata as shown below.

In [179]:
# extract the queryables endpoint

expression = parse("$.links[?(@.rel == 'http://www.opengis.net/def/rel/ogc/1.0/queryables' )].href")
r = expression.find(data_collections)
r[0].value

'https://fedeo.ceos.org/collections/PROBA.CHRIS.1A/queryables'

At this endpoint, the list of filter parameters is available as a JSON Schema.  The table below shows the filter parameters for this specific collection.

In [180]:
response = requests.get(r[0].value)   
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

,key,type,format
0,acquisitionType,string,-
5,availabilityTime,string,-
13,frame,string,-
1,illuminationAzimuthAngle,number,-
11,illuminationElevationAngle,number,-
2,instrument,string,-
7,modificationDate,string,date-time
4,orbitNumber,integer,-
3,platform,string,-
6,platformSerialIdentifier,string,-


## Search request

The WGISS CDA support both searching for collections through the IDN or FedEO and for granules in a specific collection at one of the data partners. It executes a collection or inventory search (a.k.a. granule search), as appropriate, and returns the matching results.

## Search response

The STAC API returns the search responses in (Geo)JSON formats.

The GeoJSON response elements include:

- numberReturned: number of results returned in current response page
- numberMatched: number of total results


In [185]:
print(f"{results.matched()} items found.")

9 items found.


In [182]:
data_granules

{'type': 'FeatureCollection',
 'features': [{'stac_version': '1.0.0',
   'assets': {'thumbnail': {'roles': ['thumbnail'],
     'href': 'http://tpm-ds.eo.esa.int/oads/meta/PROBA1-CHRIS/thumbnail/PR1_OPER_CHR_MO1_1P_20170419T135800_N37-075_E015-001_0001.SIP.ZIP_TIMG.jpg',
     'type': 'image/jpeg',
     'title': 'THUMBNAIL'},
    'enclosure': {'roles': ['data'],
     'href': 'https://tpm-ds.eo.esa.int/oads/data/PROBA1-CHRIS/PR1_OPER_CHR_MO1_1P_20170419T135800_N37-075_E015-001_0001.SIP.ZIP',
     'type': 'application/zip',
     'title': 'Download',
     'file:size': 257077674},
    'metadata_ogc_10_157r4': {'roles': ['metadata'],
     'href': 'https://fedeo.ceos.org/collections/PROBA.CHRIS.1A/items/PR1_OPER_CHR_MO1_1P_20170419T135800_N37-075_E015-001_0001?httpAccept=application/gml%2Bxml&recordSchema=om',
     'title': 'OGC 10-157r4 metadata',
     'type': 'application/gml+xml;profile="http://www.opengis.net/spec/EOMPOM/1.1"'},
    'metadata_ogc_17_069r3': {'roles': ['metadata'],
     'hr

```{index} double: pystac_client ; intersects
```

**Example: 2.1**  
>  Search granules by geometry {intersects} [[RD11]](#RD11) and `GET` method.  Geometry parameter can be provided as dictionary or string.

```{index} double: response element ; numberMatched (STAC)
```
The total number of results available is reported in the `numberMatched` property.

In [ ]:
print(f"{results.matched()} items found.")

```{index} double: STAC API ; POST
```

**Example: 2.2**  
>  Search granules by geometry {intersects} [[RD11]](#RD11) and `POST` method.  Geometry parameter can be provided as dictionary or string.

In [ ]:
# same request with POST
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'POST',        
    max_items = 2,
    collections=[COLLECTION_ID1],
    # intersects = json.dumps(aoi_as_dict), 
    intersects = aoi_as_dict,
    datetime=['2015-01-01T00:00:00Z', '2022-01-02T00:00:00Z']
)

In [ ]:
curl_str = curl_command(results.url_with_parameters(), "POST")
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

```{index} double: STAC API ; bbox
```

### Search by bounding box

```{index} double: pystac_client ; bbox
```

The geometry parameter can be provided as Python list or tuple.

**Example: 2.3**  
>  Search granules by bounding box {bbox} list [[RD11]](#RD11).  bbox parameter is provided as Python list.

In [ ]:
from pystac_client import Client
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items=10,
    collections=[COLLECTION_ID1],
    bbox = [14.90, 37.700, 14.99, 37.780], # Mount Etna
    # datetime=['2015-01-01T00:00:00Z', '2022-01-02T00:00:00Z']
)

Same request using `curl`.

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
display_previews(results)

In [ ]:
display_gdf_plot(results)

In [ ]:
display_map(results)

In [ ]:
display_date_distribution(results)

**Example: 2.4**  
>  Search granules by bounding box {bbox} [[RD11]](#RD11).  Geometry parameter is provided as Python tuple.

In [ ]:
# x, y = (14.95, 37.74)   # Center point of query (Mount Etna)
x, y = (4.38, 51.25)   # Center point of query (Antwerp harbour) 

r = 0.1
box = (x - r, y - r, x + r, y + r)

from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items=10,
    collections=[COLLECTION_ID1],
    bbox = box
)

Same request using `curl`.

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
display_previews(results)

In [ ]:
display_gdf_plot(results)

In [ ]:
display_map(results)

**Example: 2.5**  
>  Search granules by bounding box (`bbox`) and generate density map. 

In [ ]:
collection_id = COLLECTION_ID1

def get_results(bbox):
    x, y, x2, y2 = bbox
    results = api.search(
            method = 'GET',   
            max_items=1, 
            bbox = [x, y, x2, y2], 
            collections=[collection_id]
    ) 
    return results.matched()

collection_size = get_results([-180, -90, 180, 90])

In [ ]:
n_rows = 18
n_columns = 36

dy = 180.0 / n_rows
dx = 360.0 / n_columns
shape = (n_rows, n_columns)
Z = np.zeros(shape)

bboxes = []
for col in range(n_columns):
    for row in range(n_rows):
        x = col * dx - 180.0
        y = row * dy - 90.0
        bboxes.append((x, y, x+dx, y+dy))

In [ ]:
%%time
executor = ThreadPoolExecutor(max_workers=32)

results = executor.map(get_results, bboxes)

for col in range(n_columns):
    for row in range(n_rows):
        count = next(results)
        Z[row, col] = count

In [ ]:
print(f'Display number of granules as density map of {n_rows} rows ({dy}°) by {n_columns} columns ({dx}°).')

In [ ]:
# Get world map data from Geopandas
# worldmap = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Create axes and plot world map
fig, ax = plt.subplots(figsize=(12, 6))
# worldmap.plot(color="lightgrey", ax=ax)

side_x = np.linspace(-180, +180, n_columns)
side_y = np.linspace(-90, +90, n_rows)
X, Y = np.meshgrid(side_x, side_y)
# Z was computed before.
plt.pcolormesh(X, Y, Z, shading='auto', alpha=0.6)
plt.colorbar(label='Granules')

# Create axis limits and title
plt.xlim([-180, 180])
plt.ylim([-90, 90])

plt.title("Density Plot - " + collection_id + " (size: "+str(collection_size)+")")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

```{index} double: STAC API ; datetime
```
```{index} double: pystac_client ; datetime
```

### Search by temporal extent

**Example: 2.6**  
>  Search granules by date range (datetime) [[RD01]](#RD01).  

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items = 50,
    collections=[COLLECTION_ID1],
    datetime=['2019-01-01T00:00:00Z', '2019-12-02T00:00:00Z']
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
display_date_distribution(results)

**Example: 2.7**  
>  Search granules by open-ended date range (datetime) [[RD01]](#RD01).  

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items = 50,
    collections=[COLLECTION_ID1],
    datetime=['2021-12-01T00:00:00Z', None]
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
# keep id of first granule for future use below.
items = list(results.items())
granule_id1 = items[0].id

In [ ]:
display_date_distribution(results)

In [ ]:
display_value_distribution(results, 'sar:product_type')

```{index} double: STAC API ; ids
```

### Search by identifier


```{index} double: pystac_client ; ids
```
**Example: 2.8**  
>  Search granule by identifier (ids) [[RD01]](#RD01).  

In [ ]:
granule_id1

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    collections=[COLLECTION_ID1],
    ids=[granule_id1]
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
items = list(results.items())
print(f"{results.matched()} items found.")
assert results.matched() == 1

# Convert STAC items into data frame
stac_json = results.item_collection_as_dict()
gdf = gpd.GeoDataFrame.from_features(stac_json, "epsg:4326")
gdf.transpose()

```{index} double: STAC API ; filter
```

### Search with filter

```{index} double: pystac_client ; filter
```

**Example: 2.10**  
>  Search granules with filter {filter} [[RD01]](#RD01).  Available filters are advertised in `Queryables` object at /collections/{id}/queryables.

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items=10,
    collections=[COLLECTION_ID1],
    bbox = [14.90, 37.700, 14.99, 37.780], # Mount Etna
    datetime=['2015-01-01T00:00:00Z', '2022-01-02T00:00:00Z'],
    filter="productType='CHR_MO2_1P' and instrument='CHRIS'"
)

Same request with `curl`.

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
display_previews(results)

In [ ]:
display_gdf_plot(results)

In [ ]:
display_map(results)

In [ ]:
display_value_distribution(results, 'sar:product_type')

### Search by cloud cover

**Example: 2.11**  
>  Search granules by cloudcover (`filter` and `cloudCover`) [[RD01]](#RD01).  Available filters are advertised in `Queryables` object at /collections/{id}/queryables.

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE)  

results = api.search(
    method = 'GET',   
    max_items=50,
    collections=[COLLECTION_ID3_CLOUDS],
    filter="cloudCover < 10"    
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
# Display cloud-cover values as histogram to show that range is taken into account
stac_json = results.item_collection_as_dict()
gdf = gpd.GeoDataFrame.from_features(stac_json)
try:
  _ = gdf[['title','eo:cloud_cover']].hist()
except:
  print("eo:cloud_cover information is not available.")

In [ ]:
# fails if properties are not in the metadata.
try:
  # _ = gdf[['view:sun_elevation','view:incidence_angle','view:sun_azimuth']].plot.hist(alpha=0.7)
  _ = gdf[['view:sun_elevation','view:sun_azimuth']].plot.hist(alpha=0.7)
except:
  print("acquisition angle information is not available.")

In [ ]:
# gdf

In [ ]:
# display_value_distribution(results, 'sat:orbit_state')
display_value_distribution(results, 'sar:product_type')

### Search multiple collections

**Example: 2.12**  
>  Search granules in multiple collections {collections} [[RD01]](#RD01).  

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'GET',   
    max_items=10,
    collections=[COLLECTION_ID2_CLOUDS, COLLECTION_ID1],
    bbox = [13.90, 36.700, 15.99, 38.780], # Mount Etna (large)
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

**Example: 2.13**  
>  Search granules in multiple collections {collections} [[RD01]](#RD01) using `POST`. 

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE) 

results = api.search(
    method = 'POST',    
    max_items=50,
    collections=[COLLECTION_ID2_CLOUDS, COLLECTION_ID1],
    bbox = [13.90, 36.700, 15.99, 38.780] # Mount Etna (large)
)

In [ ]:
curl_str = curl_command(results.url_with_parameters(),'POST')
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

## Granule properties

Granules are returned via `item` links in the Catalog or Collection objects, or via the STAC API (Feature).
 An item is a GeoJSON `Feature` and the encoding is derived from the original OGC 17-003r2 encoding 
 according to a [documented mapping](https://github.com/stac-utils/stac-crosswalks/tree/master/OGC_17-003r2).
     
 The properties available include attributes from STAC extensions as well:    
  
 * [Item fields](https://github.com/radiantearth/stac-spec/blob/master/item-spec/item-spec.md#item-fields) 
 * [Common metadata elements](https://github.com/radiantearth/stac-spec/blob/master/item-spec/common-metadata.md) 
 * [EO Extension](https://github.com/stac-extensions/eo)
 * [SAR Extension](https://github.com/stac-extensions/sar)
 * [SAT Extension](https://github.com/stac-extensions/sat)
 * [Scientific Extension](https://github.com/stac-extensions/scientific)   
 * [Version Extension](https://github.com/stac-extensions/version)
 * [View Extension](https://github.com/stac-extensions/view)
 * [Projection Extension](https://github.com/stac-extensions/projection)
 * [Timestamps Extension](https://github.com/stac-extensions/timestamps)
 * [Landsat Extension](https://landsat.usgs.gov/stac/landsat-extension/schema.json)   



```{index} double: pystac_client ; assets
```
```{index} double: STAC API ; assets 
```
```{index} double: STAC API ; thumbnail 
```
```{index} double: STAC API ; data 
```
```{index} double: STAC API ; metadata 
```
```{index} double: assets ; OGC 10-157r4 
```
```{index} double: assets ; OGC 17-003r2 
```

### Assets

Granules provide access to a dictionary with `assets`.  The `roles` attribute indicates the purpose of the asset. The `href` attribute provides the URL to access the asset.  Granule assets include `thumbnail` (when available), a `data` download link (equivalent to the rel=`enclosure`), and various `metadata` formats.

The table below list some frequently used `metadata` formats and their corresponding media type (`type`).

| Format                   | type |   
| --------                   | --------- | 
| [ISO19139](https://www.iso.org/standard/32557.html)        | application/vnd.iso.19139+xml |  
| [ISO19139-2](https://www.iso.org/standard/57104.html)      | application/vnd.iso.19139-2+xml | 
| [ISO19115-3](https://www.iso.org/standard/32579.html)      | application/vnd.iso.19115-3+xml | 
| [OGC 10-157r4](https://docs.opengeospatial.org/is/10-157r4/10-157r4.html)  | application/gml+xml;profile=http://www.opengis.net/spec/EOMPOM/1.1  |
| [OGC 17-003r2](https://docs.opengeospatial.org/is/17-003r2/17-003r2.html)  | application/geo+json;profile=http://www.opengis.net/spec/eo-geojson/1.0  |

In [ ]:
# Show assets of first search result (GeoJSON)
data = results.item_collection_as_dict()
jstr = json.dumps(data['features'][1]['assets'], indent=3)
md("```json\n" + jstr + "\n```\n")

In [ ]:
df = pd.DataFrame(columns=['roles', 'title', 'type'])
    
# Display assets belonging to first item in results
for item in results.items():
    assets = item.assets
    for key in assets:     
        ndf = pd.DataFrame({ 
            'roles': assets[key].roles, 
            'type': assets[key].media_type, 
            'title': assets[key].title, 
            # 'href': assets[key].href  
        }, index = [0])
        df = pd.concat([df, ndf], ignore_index=True)
    
    break
df

## Advanced topics

```{index} double: STAC API ; conformsTo
```

### Conformance classes

The conformance classes supported by the STAC interface are advertised in the `conformsTo` property of the landing page.

In [ ]:
response = requests.get(URL_LANDING_PAGE)

data = json.loads(response.text)
jstr = json.dumps(data['conformsTo'], indent=3)
md("```json\n" + jstr + "\n```\n")

### Additional search parameters


In [ ]:
md(f'Additional search parameters beyond the STAC search parameters can be used to filter collection search results. The available parameters for collection search are advertised at {URL_LANDING_PAGE + "collections/queryables"} and represented as a JSON Schema.')

In [ ]:
URL_QUERYABLES = URL_LANDING_PAGE + 'collections/queryables'

In [ ]:
curl_str = curl_command(URL_QUERYABLES)
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
response = requests.get(URL_QUERYABLES)   
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

In [ ]:
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

Additional search parameters beyond the STAC search parameters can be used to filter granule search results.  The available parameters for granule search are advertised for each individual collection and represented as a JSON Schema.

In [ ]:
URL_COLLECTION_QUERYABLES = URL_LANDING_PAGE + 'collections/' + COLLECTION_ID1 + '/queryables'

md(f"For example, the collection `{COLLECTION_ID1}`, advertises its search parameters \
at {URL_COLLECTION_QUERYABLES} in JSON Schema format. Therefore, the following parameters can be used within a filter expression.")

Get filter parameters for granule search

In [ ]:
curl_str = curl_command(URL_COLLECTION_QUERYABLES)
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
response = requests.get(URL_COLLECTION_QUERYABLES)   
data = json.loads(response.text)    
df = pd.DataFrame(data['properties'].items(),columns=['key','value'])
df['type'] = df.apply(lambda row : row['value']['type'], axis = 1)
df['format'] = df.apply(lambda row : row['value']['format'] if 'format' in row['value'] else '-' , axis = 1)
df.drop('value',axis=1).sort_values(by=['key'])

In [ ]:
jstr = json.dumps(data, indent=3)
md("```json\n" + jstr + "\n```\n")

### CQL filter expressions

```{index} double: STAC API ; cql-text
```

The STAC interface supports the `filter` parameter and filter expressions in `cql-text` filter format at the following endpoints:

- /collections
- /collections/{collection-id}/items
- /search

At the `/search` endpoint, it is required that a single collection can be determined from the `collections` or `ids` parameter.  The queryables allowed in the filter expression are then identical to the ones at the corresponding `/collections/{collection-id}/items/queryables` endpoint.  `filter` cannot be used at the `/search` endpoint when `collections` contains 0 or more than 1 collection identifiers.

Filter expressions are to be expressed with the Text encoding of the Basic Common Query Language (Basic CQL2-Text) [[RD22]](#RD22).
See the [OGC API Features "Conformance class Filter"](conformance-class-filter) section for CQL2 examples.

**Example: 8.1**  
>  CQL Filter for collection search with logical operators (and, or).

In [ ]:
results = api.collection_search(
    filter = "platform = 'Envisat' and ( instrument = 'MERIS' or instrument = 'MIPAS' ) and organisationName = 'ESA/ESRIN'"
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=['collections'])
df[['id', 'title']]

**Example: 8.2**  
>  CQL filter for granule search with comparison operators.  Search granules with cloudCover between 10 and 15%. 

In [ ]:
from pystac_client import Client 
api = Client.open(URL_LANDING_PAGE)  

results = api.search(
    method = 'GET',   
    max_items = 30,
    collections = [COLLECTION_ID3_CLOUDS],
    filter = "cloudCover >= 10 and cloudCover < 15"   
)

In [ ]:
curl_str = curl_command(results.url_with_parameters())
md("```shell\n" + curl_str + "\n```\n")

In [ ]:
print(f"{results.matched()} items found.")

In [ ]:
# Display cloud-cover values as histogram to show that range is taken into account
stac_json = results.item_collection_as_dict()
gdf = gpd.GeoDataFrame.from_features(stac_json)
try:
  _ = gdf[['title','eo:cloud_cover']].hist()
except:
  print("eo:cloud_cover information is not available.")

## Further Reading

| **ID**  | **Title** | 
| -------- | --------- | 
| `RD11` <a name="RD11"></a> | [STAC API - Item Search](https://github.com/radiantearth/stac-api-spec/tree/main/item-search) |
| `RD12` <a name="RD12"></a> | [STAC API - Filter Extension](https://github.com/stac-api-extensions/filter) |
| `RD13` <a name="RD13"></a> | [STAC Catalog Specification](https://github.com/radiantearth/stac-spec/blob/master/catalog-spec/catalog-spec.md) | 
| `RD14` <a name="RD14"></a> | [STAC Collection Specification](https://github.com/radiantearth/stac-spec/blob/master/collection-spec/collection-spec.md) | 
| `RD15` <a name="RD15"></a>| [STAC API Specification](https://github.com/radiantearth/stac-api-spec)  | 
| `RD16` <a name="RD16"></a> | [STAC Item Specification](https://github.com/radiantearth/stac-spec/tree/master/item-spec)   | 
| `RD17` <a name="RD17"></a> | [PySTAC Documentation](https://pystac.readthedocs.io/en/stable/) | 
| `RD18` <a name="RD18"></a> | [PySTAC Client Usage](https://pystac-client.readthedocs.io/en/stable/usage.html) | 
| `RD19` <a name="RD19"></a> | [ODC STAC - Plot STAC Items on a map ](https://odc-stac.readthedocs.io/en/latest/notebooks/stac-load-e84-aws.html#Plot-STAC-Items-on-a-Map) | 
| `RD20` <a name="RD20"></a> | [OGC17-069r3, OGC API - Features - Part 1: Core](https://docs.opengeospatial.org/is/17-069r3/17-069r3.html) | 
| `RD21` <a name="RD21"></a> | [OGC19-079r2, OGC API - Features - Part 3: Filtering](https://docs.opengeospatial.org/is/19-079r2/19-079r2.html)  | 
| `RD22` <a name="RD22"></a> | [OGC21-065r2, Common Query Language (CQL2)](https://docs.ogc.org/is/21-065r2/21-065r2.html)  | 
| `RD23` <a name="RD23"></a> | [RFC 7946 - The GeoJSON Format](https://datatracker.ietf.org/doc/html/rfc7946) | 
| `RD24` <a name="RD24"></a>| [JSON Schema: A Media Type for Describing JSON Documents, draft-handrews-json-schema-02](https://datatracker.ietf.org/doc/html/draft-handrews-json-schema-02) |
| `RD25` <a name="RD25"></a>| [STAC API - Collection Search](https://github.com/stac-api-extensions/collection-search) |






